In [ ]:
import json
from pathlib import Path
from utils import *
import matplotlib.pyplot as plt
import os
from scipy.stats import ks_2samp, mannwhitneyu, wilcoxon

In [ ]:
base_dir = '/home/tomg1018/MAT1510-Project/tom1510_20251123_161839/tom1510_20251123_161839'
problems = []
ground_truth = {}

base_dir_path = Path(base_dir)
    
# Load the config
with open(base_dir_path / 'experiment.json', 'r') as f:
    experiment_data = json.load(f)

num_rollouts = experiment_data['config']['num_rollouts']

for problem in experiment_data['problems']:
    problems.append(str(problem['index']))
    ground_truth[problems[-1]] = problem['ground_truth']

print(ground_truth)

In [ ]:
def parse_all_paths(save_dir):
    """Load a reasoning graph and store path + forking node IDs (only forks with 2 children)."""
    save_dir = Path(save_dir)
    
    # Load graph structure
    with open(save_dir / 'graph_structure.json', 'r') as f:
        graph_data = json.load(f)
        nodes = graph_data['graph']['nodes']
            
    # DFS stack stores: (node_id, path_so_far, path_len, fork_ids_so_far)
    stack = [('0', '', 0, [])]  # node_id, accumulated text, path length, forking node IDs
    all_paths = []

    while stack:
        node_id, path_so_far, path_len, fork_ids = stack.pop()
        node = nodes[node_id]

        new_path = path_so_far + node["value"]
        new_path_len = path_len + 1

        # Copy list to avoid mutation
        new_fork_ids = fork_ids.copy()

        next_tokens = node['next_tokens']

        # Only consider forking nodes with exactly 2 children
        if node.get("is_forking_token", False) and len(next_tokens) == 2:
            new_fork_ids.append(node_id)

        if len(next_tokens) == 0:
            # Terminal path: store path + length + list of forking node IDs along the path
            all_paths.append({
                "completion": new_path,
                "length": new_path_len,
                "forking_node_ids": new_fork_ids
            })
        else:
            for next_token in next_tokens:
                stack.append((
                    str(next_token),
                    new_path,
                    new_path_len,
                    new_fork_ids
                ))

    # Write paths.json
    output_path = save_dir / "paths.json"
    with open(output_path, "w") as f:
        json.dump(all_paths, f, indent=2)


In [ ]:
def parse_all_paths(save_dir, ground_truth):
    """Load a reasoning graph and store paths + fork info + correctness + extracted answer."""
    save_dir = Path(save_dir)
    
    # Load graph structure
    with open(save_dir / 'graph_structure.json', 'r') as f:
        nodes = json.load(f)['graph']['nodes']
            
    # DFS stack stores: (node_id, path_so_far, path_len, fork_ids_so_far, child_of_fork_so_far)
    stack = [('0', '', 0, [], [])]  # node_id, text, path length, fork_ids, child_of_fork
    all_paths = []

    while stack:
        node_id, path_so_far, path_len, fork_ids, child_of_fork = stack.pop()
        node = nodes[node_id]

        new_path = path_so_far + node["value"]
        new_path_len = path_len + 1

        next_tokens = node['next_tokens']

        # Copy lists to avoid mutation
        new_fork_ids = fork_ids.copy()
        new_child_of_fork = child_of_fork.copy()

        # Only consider forking nodes with exactly 2 children
        if node.get("is_forking_token", False) and len(next_tokens) == 2:
            new_fork_ids.append(node_id)
            # child will be appended when choosing a next token below

        if len(next_tokens) == 0:
            # Terminal node: extract answer and check correctness
            extracted_answer = extract_answer(new_path)
            correct = extracted_answer == float(ground_truth)

            all_paths.append({
                "completion": new_path,
                "length": new_path_len,
                "forking_node_ids": new_fork_ids,
                "child_of_fork": new_child_of_fork,
                "extracted_answer": extracted_answer,
                "correct": correct
            })
        else:
            for next_token in next_tokens:
                next_id = str(next_token)
                updated_child_of_fork = new_child_of_fork.copy()
                # If current node is a forking node, the chosen child is this next token
                if node.get("is_forking_token", False) and len(next_tokens) == 2:
                    updated_child_of_fork.append(next_id)
                stack.append((
                    next_id,
                    new_path,
                    new_path_len,
                    new_fork_ids,
                    updated_child_of_fork
                ))

    # Write paths.json
    output_path = save_dir / "paths.json"
    with open(output_path, "w") as f:
        json.dump(all_paths, f, indent=2)

In [ ]:
# Finding all of the paths that the graph went through
base_dir = '/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006'
problem = 12
for rollout in range(25):
    cur_dir = base_dir + '/problem_' + str(problem) + '/rollout_' + str(rollout)
    parse_all_paths(cur_dir, ground_truth = 540)

In [ ]:
# Finding all of the paths that the graph went through
base_dir = '/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229'
problem = 9
for rollout in range(25):
    cur_dir = base_dir + '/problem_' + str(problem) + '/rollout_' + str(rollout)
    parse_all_paths(cur_dir, ground_truth = 116)

In [ ]:
# Finding all of the paths that the graph went through
base_dir = '/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417'
problem = 0
for rollout in range(25):
    cur_dir = base_dir + '/problem_' + str(problem) + '/rollout_' + str(rollout)
    parse_all_paths(cur_dir, ground_truth = 204)

In [ ]:
import os
import json

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"

correct = 0
incorrect = 0

for rollout_name in os.listdir(base_dir):
    rollout_path = os.path.join(base_dir, rollout_name)

    if not rollout_name.startswith("rollout_"):
        continue
    if not os.path.isdir(rollout_path):
        continue

    paths_json = os.path.join(rollout_path, "paths.json")
    if not os.path.exists(paths_json):
        continue

    with open(paths_json, "r") as f:
        data = json.load(f)

    for branch in data:
        if branch.get("correct"):
            correct += 1
        else:
            incorrect += 1

print("Question 12")
print("Correct:", correct)
print("Incorrect:", incorrect)


In [ ]:
import os
import json

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9"

correct = 0
incorrect = 0

for rollout_name in os.listdir(base_dir):
    rollout_path = os.path.join(base_dir, rollout_name)

    if not rollout_name.startswith("rollout_"):
        continue
    if not os.path.isdir(rollout_path):
        continue

    paths_json = os.path.join(rollout_path, "paths.json")
    if not os.path.exists(paths_json):
        continue

    with open(paths_json, "r") as f:
        data = json.load(f)

    for branch in data:
        if branch.get("correct"):
            correct += 1
        else:
            incorrect += 1

print("Question 9")
print("Correct:", correct)
print("Incorrect:", incorrect)


In [ ]:
import os
import json

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"

correct = 0
incorrect = 0

for rollout_name in os.listdir(base_dir):
    rollout_path = os.path.join(base_dir, rollout_name)

    if not rollout_name.startswith("rollout_"):
        continue
    if not os.path.isdir(rollout_path):
        continue

    paths_json = os.path.join(rollout_path, "paths.json")
    if not os.path.exists(paths_json):
        continue

    with open(paths_json, "r") as f:
        data = json.load(f)

    for branch in data:
        if branch.get("correct"):
            correct += 1
        else:
            incorrect += 1

print("Question 0")
print("Correct:", correct)
print("Incorrect:", incorrect)


Q0: 10.79%
Q9: 35.23%
Q12: 17.35%

In [ ]:
import json
import networkx as nx

file = "tom1510_20251123_191006/tom1510_20251123_191006/problem_12/rollout_0/paths.json"

# Load completions
with open(file, "r") as f:
    completions = json.load(f)

# Build the DAG
G = nx.DiGraph()
completion_leaf_correct = {}  # maps leaf node to correctness

for comp in completions:
    forks = comp["forking_node_ids"]
    childs = comp["child_of_fork"]
    is_correct = comp["correct"]

    # Connect nodes
    for i in range(len(forks)):
        G.add_edge(forks[i], childs[i])
        if i + 1 < len(forks):
            G.add_edge(childs[i], forks[i+1])

    # Map leaf node to correctness
    leaf = childs[-1]
    completion_leaf_correct[leaf] = is_correct

# Function to get all reachable leaves from a node
def all_reachable_leaves(node, G):
    leaves = set()
    stack = [node]
    while stack:
        n = stack.pop()
        children = list(G.successors(n))
        if not children:  # leaf
            leaves.add(n)
        else:
            stack.extend(children)
    return leaves

# Find nodes where all reachable leaves are incorrect or correct
incorrect_nodes = []
correct_nodes = []

for node in G.nodes():
    leaves = all_reachable_leaves(node, G)
    if not leaves:
        continue
    leaf_statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]

    if all(not status for status in leaf_statuses):
        incorrect_nodes.append(node)
    if all(status for status in leaf_statuses):
        correct_nodes.append(node)

print("Nodes where all reachable paths yield incorrect completions:")
print(incorrect_nodes)

print("\nNodes where all reachable paths yield correct completions:")
print(correct_nodes)

print(len(incorrect_nodes), len(correct_nodes))

In [ ]:
import json
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout
import matplotlib.pyplot as plt

for rollout in range(25):
    # Load the paths.json
    file = f"tom1510_20251123_191006/tom1510_20251123_191006/problem_12/rollout_{rollout}/paths.json"
    with open(file, "r") as f:
        completions = json.load(f)

    file2 = f"tom1510_20251123_191006/tom1510_20251123_191006/problem_12/rollout_{rollout}/graph_structure.json"
    with open(file2, "r") as f:
        graph = json.load(f)
    
    graph_key = graph["graph"]
    nodes = graph_key["nodes"]
    # Build DAG and map leaves to correctness
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i+1])

        # Last child is the leaf
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to find all reachable leaves from a node
    def all_reachable_leaves(node, G):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    # Determine node color based on correctness of all reachable leaves
    node_colors = {}
    for node in G.nodes():
        leaves = all_reachable_leaves(node, G)
        if not leaves:
            continue
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            node_colors[node] = "green"      # all correct
        elif all(not s for s in statuses):
            node_colors[node] = "red"        # all incorrect
        else:
            node_colors[node] = "gray"       # mixed

    # Layout the DAG as a hierarchical tree

    node_labels = {n: nodes[n]["value"] for n in G.nodes() if n in nodes}
    pos = graphviz_layout(G, prog="dot")  # requires pygraphviz or pydot installed

    # Draw the graph
    plt.figure(figsize=(14, 10))
    nx.draw(G, pos,
            labels=node_labels,
            with_labels=True,
            node_size=800,
            node_color=[node_colors.get(n, "gray") for n in G.nodes()],
            font_size=9,
            arrows=True,
            arrowsize=20)
    plt.title(f"Reasoning Tree with Forking Nodes (Green=Correct Path, Red=Incorrect Path, Gray=Uncertain Path) Problem 12 Rollout {rollout}")
    plt.show()


In [ ]:
import json
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout
import matplotlib.pyplot as plt

for rollout in range(25):
    # Load the paths.json
    file = f"tom1510_20251123_214229/tom1510_20251123_214229/problem_9/rollout_{rollout}/paths.json"
    with open(file, "r") as f:
        completions = json.load(f)

    file2 = f"tom1510_20251123_214229/tom1510_20251123_214229/problem_9/rollout_{rollout}/graph_structure.json"
    with open(file2, "r") as f:
        graph = json.load(f)
    
    graph_key = graph["graph"]
    nodes = graph_key["nodes"]
    # Build DAG and map leaves to correctness
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i+1])

        # Last child is the leaf
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to find all reachable leaves from a node
    def all_reachable_leaves(node, G):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    # Determine node color based on correctness of all reachable leaves
    node_colors = {}
    for node in G.nodes():
        leaves = all_reachable_leaves(node, G)
        if not leaves:
            continue
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            node_colors[node] = "green"      # all correct
        elif all(not s for s in statuses):
            node_colors[node] = "red"        # all incorrect
        else:
            node_colors[node] = "gray"       # mixed

    # Layout the DAG as a hierarchical tree

    node_labels = {n: nodes[n]["value"] for n in G.nodes() if n in nodes}
    pos = graphviz_layout(G, prog="dot")  # requires pygraphviz or pydot installed

    # Draw the graph
    plt.figure(figsize=(14, 10))
    nx.draw(G, pos,
            labels=node_labels,
            with_labels=True,
            node_size=800,
            node_color=[node_colors.get(n, "gray") for n in G.nodes()],
            font_size=9,
            arrows=True,
            arrowsize=20)
    plt.title(f"Reasoning Tree with Forking Nodes (Green=Correct Path, Red=Incorrect Path, Gray=Uncertain Path) Problem 9 Rollout {rollout}")
    plt.show()


Note: Crashes if a forking token is $$

In [ ]:
import os
import json
import networkx as nx
import matplotlib.pyplot as plt

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
rollouts = [d for d in os.listdir(base_dir) if d.startswith("rollout_")]

# Cumulative lists
entropy_green_children = []
entropy_red_children = []
entropy_gray_children = []

for rollout in rollouts:
    paths_file = os.path.join(base_dir, rollout, "paths.json")
    graph_file = os.path.join(base_dir, rollout, "graph_structure.json")

    # Load JSON files
    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)
    nodes = graph["graph"]["nodes"]

    # Build DAG and map leaves to correctness
    G = nx.DiGraph()
    completion_leaf_correct = {}
    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i+1])
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to get all reachable leaves
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    # Collect child-of-fork nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    # Determine node colors and collect entropies
    node_colors = {}
    for node in G.nodes():
        if node not in child_nodes:
            continue

        leaves = all_reachable_leaves(node)
        if not leaves:
            continue
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        metric_value = nodes.get(node, {}).get("metric_value", None)

        if all(statuses):
            node_colors[node] = "green"
            if metric_value is not None:
                entropy_green_children.append(metric_value)
        elif all(not s for s in statuses):
            node_colors[node] = "red"
            if metric_value is not None:
                entropy_red_children.append(metric_value)
        else:
            node_colors[node] = "gray"
            if metric_value is not None:
                entropy_gray_children.append(metric_value)

# Print counts
print(len(entropy_green_children), len(entropy_red_children), len(entropy_gray_children))

# Plot histograms

plt.figure(figsize=(10, 6))
if entropy_green_children:
    plt.hist(entropy_green_children, density = True, bins=20, alpha=0.6, label="Correct Paths", color="green")
if entropy_red_children:
    plt.hist(entropy_red_children, density = True, bins=20, alpha=0.6, label="Incorrect Paths", color="red")
if entropy_gray_children:
    plt.hist(entropy_gray_children, density = True, bins=20, alpha=0.6, label="Uncertain Paths", color="gray")
plt.xlabel("Entropy")
plt.ylabel("Density")
plt.title("Entropy distribution by correctness group across all rollouts")
plt.legend()
plt.show()

if entropy_green_children:
    plt.figure(figsize=(10, 6))
    plt.hist(entropy_green_children, density = True, bins=20, alpha=0.6, label="Green (all correct)", color="green")
    plt.xlabel("Entropy")
    plt.ylabel("Density")
    plt.title("Entropy distribution by correctness group across all rollouts")
    plt.show()
if entropy_red_children:
    plt.figure(figsize=(10, 6))
    plt.hist(entropy_red_children, density = True, bins=20, alpha=0.6, label="Red (all incorrect)", color="red")
    plt.xlabel("Entropy")
    plt.ylabel("Density")
    plt.title("Entropy distribution by correctness group across all rollouts")
    plt.show()
if entropy_gray_children:
    plt.figure(figsize=(10, 6))
    plt.hist(entropy_gray_children, density = True, bins=20, alpha=0.6, label="Gray (mixed)", color="gray")
    plt.xlabel("Density")
    plt.ylabel("Number of nodes")
    plt.title("Entropy distribution by correctness group across all rollouts")
    plt.show()

cats = [
    ("entropy_green_children", entropy_green_children),
    ("entropy_red_children", entropy_red_children),
    ("entropy_gray_children", entropy_gray_children),
]

for i in range(len(cats)):
    for j in range(i+1, len(cats)):
        name1, cat1 = cats[i]
        name2, cat2 = cats[j]
    
        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann-Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
import os
import json
import networkx as nx
import matplotlib.pyplot as plt

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25

# Cumulative lists
entropy_green_children = []
entropy_red_children = []
entropy_gray_children = []

for base_dir in base_dirs:    
    for rollout in range(num_rollouts):
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        # Load JSON files
        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)
        nodes = graph["graph"]["nodes"]

        # Build DAG and map leaves to correctness
        G = nx.DiGraph()
        completion_leaf_correct = {}
        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i+1])
            leaf = childs[-1]
            completion_leaf_correct[leaf] = is_correct

        # Function to get all reachable leaves
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        # Collect child-of-fork nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        # Determine node colors and collect entropies
        node_colors = {}
        for node in G.nodes():
            if node not in child_nodes:
                continue

            leaves = all_reachable_leaves(node)
            if not leaves:
                continue
            statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
            metric_value = nodes.get(node, {}).get("metric_value", None)

            if all(statuses):
                node_colors[node] = "green"
                if metric_value is not None:
                    entropy_green_children.append(metric_value)
            elif all(not s for s in statuses):
                node_colors[node] = "red"
                if metric_value is not None:
                    entropy_red_children.append(metric_value)
            else:
                node_colors[node] = "gray"
                if metric_value is not None:
                    entropy_gray_children.append(metric_value)

# Print counts
print(len(entropy_green_children), len(entropy_red_children), len(entropy_gray_children))

# Plot histograms

plt.figure(figsize=(10, 6))
if entropy_green_children:
    plt.hist(entropy_green_children, density = True, bins=20, alpha=0.6, label="Correct Paths", color="green")
if entropy_red_children:
    plt.hist(entropy_red_children, density = True, bins=20, alpha=0.6, label="Incorrect Paths", color="red")
if entropy_gray_children:
    plt.hist(entropy_gray_children, density = True, bins=20, alpha=0.6, label="Uncertain Paths", color="gray")
plt.xlabel("Entropy")
plt.ylabel("Density")
plt.title("Entropy Distributions of Forking Tokens by Correctness of Their Paths")
plt.legend()
plt.show()

if entropy_green_children:
    plt.figure(figsize=(10, 6))
    plt.hist(entropy_green_children, density = True, bins=20, alpha=1, label="Green (all correct)", color="green")
    plt.xlabel("Entropy")
    plt.ylabel("Density")
    plt.title("Histogram of Entropy of Correct Forking Tokens")
    plt.show()
if entropy_red_children:
    plt.figure(figsize=(10, 6))
    plt.hist(entropy_red_children, density = True, bins=20, alpha=1, label="Red (all incorrect)", color="red")
    plt.xlabel("Entropy")
    plt.ylabel("Density")
    plt.title("Histogram of Entropy of Incorrect Forking Tokens")
    plt.show()
if entropy_gray_children:
    plt.figure(figsize=(10, 6))
    plt.hist(entropy_gray_children, density = True, bins=20, alpha=1, label="Gray (mixed)", color="gray")
    plt.xlabel("Density")
    plt.ylabel("Density")
    plt.title("Histogram of Entropy of Uncertain Forking Tokens")
    plt.show()

cats = [
    ("entropy_green_children", entropy_green_children),
    ("entropy_red_children", entropy_red_children),
    ("entropy_gray_children", entropy_gray_children),
]

for i in range(len(cats)):
    for j in range(i+1, len(cats)):
        name1, cat1 = cats[i]
        name2, cat2 = cats[j]
    
        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann-Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25 

# Cumulative lists
prob_green_children = []
prob_red_children = []
prob_gray_children = []

for rollout in range(num_rollouts):
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    # Load JSON files
    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)
    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}
    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to classify node based on reachable leaves
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_node(node):
        leaves = all_reachable_leaves(node)
        if not leaves:
            return None
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child-of-fork nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {child: classify_node(child) for child in child_nodes}

    # Loop over all forks and collect children entropies if children have mixed classes
    for fork_node in set(f for comp in completions for f in comp["forking_node_ids"]):
        children = list(G.successors(fork_node))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children if c in node_colors]
        if len(set(child_classes)) <= 1:
            continue  # all children same class

        for i, child in enumerate(children):
            child_class = child_classes[i]
            selected_prob = nodes.get(child, {}).get("selected_prob", None)
            if selected_prob is None:
                continue
            if child_class == "green":
                prob_green_children.append(selected_prob)
            elif child_class == "red":
                prob_red_children.append(selected_prob)
            else:
                prob_gray_children.append(selected_prob)

# Plot one histogram
plt.figure(figsize=(10, 6))
if prob_green_children:
    plt.hist(prob_green_children, density = True, bins=20, alpha=0.3, label="Children Leading to Correct Paths", color="green")
if prob_red_children:
    plt.hist(prob_red_children, density = True, bins=20, alpha=0.3, label="Children Leading to Incorrect Paths", color="red")
if prob_gray_children:
    plt.hist(prob_gray_children, density = True, bins=20, alpha=0.3, label="Children Leading to Uncertain Paths", color="gray")


plt.xlim(0,1)
plt.xlabel("Token Probability")
plt.ylabel("Density of Forking Nodes")
plt.title("Distribution of Token Probabilities for Pivotal Forks")
plt.legend()
plt.show()

# Plot separate histograms
if prob_green_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_green_children, density = True, bins=20, alpha=0.6, label="Green (all correct)", color="green")
    plt.xlim(0,1)
    plt.xlabel("Token Probability")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability for Pivotal Forks Leading to Correct Paths")
    plt.show()
if prob_red_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_red_children, density = True, bins=20, alpha=0.6, label="Red (all incorrect)", color="red")
    plt.xlim(0,1)
    plt.xlabel("Token Probability")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability for Pivotal Forks Leading to Incorrect Paths")
    plt.show()
if prob_gray_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_gray_children, density = True, bins=20, alpha=0.6, label="Gray (mixed)", color="gray")
    plt.xlim(0,1)
    plt.xlabel("Token Probability")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability for Pivotal Forks Leading to Uncertain Paths")
    plt.show()


cats_prob = [
    ("prob_green_children", prob_green_children),
    ("prob_red_children", prob_red_children),
    ("prob_gray_children", prob_gray_children),
]

for i in range(len(cats_prob)):
    for j in range(i+1, len(cats_prob)):
        name1, cat1 = cats_prob[i]
        name2, cat2 = cats_prob[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann-Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25 

# Cumulative lists
prob_green_children = []
prob_red_children = []
prob_gray_children = []

for base_dir in base_dirs:
    for rollout in range(num_rollouts):
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        # Load JSON files
        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)
        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}
        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])
            leaf = childs[-1]
            completion_leaf_correct[leaf] = is_correct

        # Function to classify node based on reachable leaves
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                   stack.extend(children)
            return leaves

        def classify_node(node):
            leaves = all_reachable_leaves(node)
            if not leaves:
                return None
            statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child-of-fork nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {child: classify_node(child) for child in child_nodes}

        # Loop over all forks and collect children entropies if children have mixed classes
        for fork_node in set(f for comp in completions for f in comp["forking_node_ids"]):
            children = list(G.successors(fork_node))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children if c in node_colors]
            if len(set(child_classes)) <= 1:
                continue  # all children same class

            for i, child in enumerate(children):
                child_class = child_classes[i]
                selected_prob = nodes.get(child, {}).get("selected_prob", None)
                if selected_prob is None:
                    continue
                if child_class == "green":
                    prob_green_children.append(selected_prob)
                elif child_class == "red":
                    prob_red_children.append(selected_prob)
                else:
                    prob_gray_children.append(selected_prob)


# Plot one histogram
plt.figure(figsize=(10, 6))
if prob_green_children:
    plt.hist(prob_green_children, density = True, bins=20, alpha=0.3, label="Children Leading to Correct Paths", color="green")
if prob_red_children:
    plt.hist(prob_red_children, density = True, bins=20, alpha=0.3, label="Children Leading to Incorrect Paths", color="red")
if prob_gray_children:
    plt.hist(prob_gray_children, density = True, bins=20, alpha=0.3, label="Children Leading to Uncertain Paths", color="gray")


plt.xlim(0,1)
plt.xlabel("Token Probability")
plt.ylabel("Density of Forking Nodes")
plt.title("Distribution of Token Probabilities for Pivotal Forks")
plt.legend()
plt.show()

# Plot separate histograms
if prob_green_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_green_children, density = True, bins=20, alpha=0.6, label="Green (all correct)", color="green")
    plt.xlim(0,1)
    plt.xlabel("Token Probability")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability for Pivotal Forks Leading to Correct Paths")
    plt.show()
if prob_red_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_red_children, density = True, bins=20, alpha=0.6, label="Red (all incorrect)", color="red")
    plt.xlim(0,1)
    plt.xlabel("Token Probability")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability for Pivotal Forks Leading to Incorrect Paths")
    plt.show()
if prob_gray_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_gray_children, density = True, bins=20, alpha=0.6, label="Gray (mixed)", color="gray")
    plt.xlim(0,1)
    plt.xlabel("Token Probability")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability for Pivotal Forks Leading to Uncertain Paths")
    plt.show()


cats_prob = [
    ("prob_green_children", prob_green_children),
    ("prob_red_children", prob_red_children),
    ("prob_gray_children", prob_gray_children),
]

for i in range(len(cats_prob)):
    for j in range(i+1, len(cats_prob)):
        name1, cat1 = cats_prob[i]
        name2, cat2 = cats_prob[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann-Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25  # or adjust based on your data

# Cumulative lists
prob_index_green_children = []
prob_index_red_children = []
prob_index_gray_children = []

for rollout in range(num_rollouts):
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    # Load JSON files
    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)
    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}
    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to classify node based on reachable leaves
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_node(node):
        leaves = all_reachable_leaves(node)
        if not leaves:
            return None
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child-of-fork nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {child: classify_node(child) for child in child_nodes}

    # Loop over all forks and collect children entropies if children have mixed classes
    for fork_node in set(f for comp in completions for f in comp["forking_node_ids"]):
        children = list(G.successors(fork_node))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children if c in node_colors]
        if len(set(child_classes)) <= 1:
            continue  # all children same class

        for i, child in enumerate(children):
            child_class = child_classes[i]

            node_info = nodes.get(child, {})
            selected_prob = node_info.get("selected_prob", None)
            topk_probs = node_info.get("topk_probs", None)
    
            # Skip if either is missing
            if selected_prob is None or topk_probs is None:
                continue

            # Find the index of the selected_prob in topk_probs
            try:
                selected_index = topk_probs.index(selected_prob)
            except ValueError:
                # If not found exactly, choose closest
                selected_index = min(range(len(topk_probs)), key=lambda j: abs(topk_probs[j]-selected_prob))

            # Append index instead of probability
            if child_class == "green":
                prob_index_green_children.append(selected_index+1)
            elif child_class == "red":
                prob_index_red_children.append(selected_index+1)
            else:
                prob_index_gray_children.append(selected_index+1)

plt.figure(figsize=(10,6))
if prob_index_green_children:
    plt.hist(prob_index_green_children, density = True, bins=25, alpha=0.3, label="Children Leading to Correct Paths", color="green")
if prob_index_red_children:
    plt.hist(prob_index_red_children, density = True, bins=25, alpha=0.3, label="Children Leading to Incorrect Paths", color="red")
if prob_index_gray_children:
    plt.hist(prob_index_gray_children, density = True, bins=25, alpha=0.3, label="Children Leading to Uncertain Paths", color="gray")

plt.xlim(1,25)
plt.xlabel("kth most likely token")
plt.ylabel("Density of Forking Nodes")
plt.title("Distribution of Token Probability Indices for Pivotal Forks")
plt.legend()
plt.show()

# Plot separate histograms
if prob_index_green_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_index_green_children, density = True, bins=25, alpha=0.6, label="Green (all correct)", color="green")
    plt.xlim(1,25)
    plt.xlabel("kth most likely token")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability Indices for Pivotal Forks Leading to Correct Paths")
    plt.legend()
    plt.show()
if prob_index_red_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_index_red_children, density = True, bins=25, alpha=0.6, label="Red (all incorrect)", color="red")
    plt.xlim(1,25)
    plt.xlabel("kth most likely token")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability Indices for Pivotal Forks Leading to Incorrect Paths")
    plt.legend()
    plt.show()
if prob_index_gray_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_index_gray_children, density = True, bins=25, alpha=0.6, label="Gray (mixed)", color="gray")
    plt.xlim(1,25)
    plt.xlabel("kth most likely token")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability Indices for Pivotal Forks Leading to Uncertain Paths")
    plt.show()

cats_prob_index = [
    ("prob_index_green_children", prob_index_green_children),
    ("prob_index_red_children", prob_index_red_children),
    ("prob_index_gray_children", prob_index_gray_children),
]

for i in range(len(cats_prob_index)):
    for j in range(i+1, len(cats_prob_index)):
        name1, cat1 = cats_prob_index[i]
        name2, cat2 = cats_prob_index[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann-Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
#All rollouts

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25  # or adjust based on your data

# Cumulative lists
prob_index_green_children = []
prob_index_red_children = []
prob_index_gray_children = []

for base_dir in base_dirs: 
    for rollout in range(num_rollouts):
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        # Load JSON files
        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)
        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}
        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])
            leaf = childs[-1]
            completion_leaf_correct[leaf] = is_correct

        # Function to classify node based on reachable leaves
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_node(node):
            leaves = all_reachable_leaves(node)
            if not leaves:
                return None
            statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child-of-fork nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {child: classify_node(child) for child in child_nodes}

        # Loop over all forks and collect children entropies if children have mixed classes
        for fork_node in set(f for comp in completions for f in comp["forking_node_ids"]):
            children = list(G.successors(fork_node))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children if c in node_colors]
            if len(set(child_classes)) <= 1:
                continue  # all children same class

            for i, child in enumerate(children):
                child_class = child_classes[i]

                node_info = nodes.get(child, {})
                selected_prob = node_info.get("selected_prob", None)
                topk_probs = node_info.get("topk_probs", None)
    
                # Skip if either is missing
                if selected_prob is None or topk_probs is None:
                    continue

                # Find the index of the selected_prob in topk_probs
                try:
                    selected_index = topk_probs.index(selected_prob)
                except ValueError:
                    # If not found exactly, choose closest
                    selected_index = min(range(len(topk_probs)), key=lambda j: abs(topk_probs[j]-selected_prob))

                # Append index instead of probability
                if child_class == "green":
                    prob_index_green_children.append(selected_index+1)
                elif child_class == "red":
                    prob_index_red_children.append(selected_index+1)
                else:
                    prob_index_gray_children.append(selected_index+1)

plt.figure(figsize=(10,6))
if prob_index_green_children:
    plt.hist(prob_index_green_children, density = True, bins=25, alpha=0.3, label="Children Leading to Correct Paths", color="green")
if prob_index_red_children:
    plt.hist(prob_index_red_children, density = True, bins=25, alpha=0.3, label="Children Leading to Incorrect Paths", color="red")
if prob_index_gray_children:
    plt.hist(prob_index_gray_children, density = True, bins=25, alpha=0.3, label="Children Leading to Uncertain Paths", color="gray")

plt.xlim(1,25)
plt.xlabel("kth most likely token")
plt.ylabel("Density of Forking Nodes")
plt.title("Distribution of Token Probability Indices for Pivotal Forks")
plt.legend()
plt.show()

# Plot separate histograms
if prob_index_green_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_index_green_children, density = True, bins=25, alpha=0.6, label="Green (all correct)", color="green")
    plt.xlim(1,25)
    plt.xlabel("kth most likely token")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability Indices for Pivotal Forks Leading to Correct Paths")
    plt.legend()
    plt.show()
if prob_index_red_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_index_red_children, density = True, bins=25, alpha=0.6, label="Red (all incorrect)", color="red")
    plt.xlim(1,25)
    plt.xlabel("kth most likely token")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability Indices for Pivotal Forks Leading to Incorrect Paths")
    plt.legend()
    plt.show()
if prob_index_gray_children:
    plt.figure(figsize=(10, 6))
    plt.hist(prob_index_gray_children, density = True, bins=25, alpha=0.6, label="Gray (mixed)", color="gray")
    plt.xlim(1,25)
    plt.xlabel("kth most likely token")
    plt.ylabel("Density of Forking Nodes")
    plt.title("Distribution of Token Probability Indices for Pivotal Forks Leading to Uncertain Paths")
    plt.show()

cats_prob_index = [
    ("prob_index_green_children", prob_index_green_children),
    ("prob_index_red_children", prob_index_red_children),
    ("prob_index_gray_children", prob_index_gray_children),
]

for i in range(len(cats_prob_index)):
    for j in range(i+1, len(cats_prob_index)):
        name1, cat1 = cats_prob_index[i]
        name2, cat2 = cats_prob_index[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann-Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25  # adjust based on your data

# Cumulative lists
greengray_probs = []  # now storing index (1 = top-1, etc.)
red_probs = []

# For paired t-test
paired_probs_greengray = []
paired_probs_red = []

for rollout in range(num_rollouts):
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    # Load JSON files
    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)
    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}
    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_node(node):
        leaves = all_reachable_leaves(node)
        if not leaves:
            return None
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {child: classify_node(child) for child in child_nodes}

    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

    for fork_node in fork_nodes:
        children = list(G.successors(fork_node))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children if c in node_colors]

        # Only forks with mixed red vs green/gray
        if "red" not in child_classes:
            continue
        if not any(c in {"green", "gray"} for c in child_classes):
            continue
        if len(set(child_classes)) == 1:
            continue

        # Collect probability index instead of probability
        greengray_vals = []
        red_vals = []

        for i, child in enumerate(children):
            cls = child_classes[i]

            node_info = nodes.get(child, {})
            selected_prob = node_info.get("selected_prob")
            topk_probs = node_info.get("topk_probs")

            if selected_prob is None or topk_probs is None:
                continue

            # Determine rank/index
            try:
                idx = topk_probs.index(selected_prob)
            except ValueError:
                idx = min(range(len(topk_probs)), key=lambda j: abs(topk_probs[j] - selected_prob))

            index_value = idx + 1  # convert 0-based → 1-based

            if cls in {"green", "gray"}:
                greengray_probs.append(index_value)
                greengray_vals.append(index_value)
            elif cls == "red":
                red_probs.append(index_value)
                red_vals.append(index_value)

        # Save paired index values (one per fork)
        if red_vals and greengray_vals:
            paired_probs_red.append(red_vals[0])
            paired_probs_greengray.append(greengray_vals[0])

# -------- Plotting --------

plt.figure(figsize=(10, 6))
plt.hist(greengray_probs, bins=20, alpha=0.6, label="Correct/Unclear", color="Green", density=True)
plt.hist(red_probs, bins=20, alpha=0.6, label="Incorrect", color="Red", density=True)
plt.xlabel("k-th Most Likely Token")
plt.ylabel("Density")
plt.title("Token Probability Index for Correct/Unclear vs Incorrect Children at Impactful Forks")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(greengray_probs, bins=20, alpha=0.6, label="Green/Gray", color="Blue", density=True)
plt.xlabel("k-th most likely token")
plt.ylabel("Density")
plt.title("Probability Index for Green/Gray Children")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(red_probs, bins=20, alpha=0.6, label="Red", color="Gray", density=True)
plt.xlabel("k-th most likely token")
plt.ylabel("Density")
plt.title("Probability Index for Red Children")
plt.show()

cats_probs_pivotal = [
    ('greengray_probs', greengray_probs),
    ('red_probs', red_probs)
]

for i in range(len(cats_probs_pivotal)):
    for j in range(i + 1, len(cats_probs_pivotal)):
        name1, cat1 = cats_probs_pivotal[i]
        name2, cat2 = cats_probs_pivotal[j]

        wilcoxonstat, wilcoxonp_value = wilcoxon(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Wilcoxon: {wilcoxonstat}, p={wilcoxonp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25  # adjust based on your data

# Cumulative lists
greengray_probs = []
red_probs = []

# For paired t-test
paired_probs_greengray = []
paired_probs_red = []

for rollout in range(num_rollouts):
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    # Load JSON files
    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)
    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}
    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to classify node based on reachable leaves
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_node(node):
        leaves = all_reachable_leaves(node)
        if not leaves:
            return None
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child-of-fork nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {child: classify_node(child) for child in child_nodes}

    # Loop over forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])
    for fork_node in fork_nodes:
        children = list(G.successors(fork_node))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children if c in node_colors]
        # Only consider forks with one red and at least one green/gray
        if "red" not in child_classes:
            continue
        if not any(c in {"green", "gray"} for c in child_classes):
            continue
        if len(set(child_classes)) == 1:
            continue

        for i, child in enumerate(children):
            cls = child_classes[i]
            selected_prob = nodes.get(child, {}).get("selected_prob", None)
            if selected_prob is None:
                continue
            # Combine green+gray
            if cls in {"green", "gray"}:
                greengray_probs.append(selected_prob)
            elif cls == "red":
                red_probs.append(selected_prob)

        # Save paired probabilities for t-test
        red_child_probs = [nodes[c].get("selected_prob") for i, c in enumerate(children)
                           if child_classes[i] == "red" and nodes[c].get("selected_prob") is not None]
        greengray_child_probs = [nodes[c].get("selected_prob") for i, c in enumerate(children)
                                 if child_classes[i] in {"green", "gray"} and nodes[c].get("selected_prob") is not None]

        # Use only one value per fork (e.g., mean if multiple children in same class)
        if red_child_probs and greengray_child_probs:
            paired_probs_red.append(red_child_probs[0])  # or np.mean(red_child_probs)
            paired_probs_greengray.append(greengray_child_probs[0])  # or np.mean(greengray_child_probs)

# Histogram
plt.figure(figsize=(10, 6))
plt.hist(greengray_probs, bins=20, alpha=0.6, label="Correct/Unclear", color="Green", density=True)
plt.hist(red_probs, bins=20, alpha=0.6, label="Incorrect", color="Red", density=True)
plt.xlabel("Selected Token Probability")
plt.ylabel("Density")
plt.title("Token Probabilities for Correct/Unclear and Incorrect Children of Impactful Forks")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(greengray_probs, bins=20, alpha=0.6, label="Green/Gray", color="Blue", density=True)
plt.xlabel("Selected Token Probability")
plt.ylabel("Density")
plt.title("Token probabilities for Impactful Forks")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(red_probs, bins=20, alpha=0.6, label="Red", color="Gray", density=True)
plt.xlabel("Selected Token Probability")
plt.ylabel("Density")
plt.title("Token probabilities for Non Impactful Forks")
plt.show()

cats_probs_pivotal = [
    ('greengray_probs', greengray_probs),
    ('red_probs', red_probs)
]

for i in range(len(cats_probs_pivotal)):
    for j in range(i+1, len(cats_probs_pivotal)):
        name1, cat1 = cats_probs_pivotal[i]
        name2, cat2 = cats_probs_pivotal[j]

        wilcoxonstat, wilcoxonp_value = wilcoxon(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Wilcoxon: {wilcoxonstat}, p={wilcoxonp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12","/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25  # adjust based on your data

# Cumulative lists
greengray_probs = []
red_probs = []

# For paired t-test
paired_probs_greengray = []
paired_probs_red = []

for base_dir in base_dirs:
    for rollout in range(num_rollouts):
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        # Load JSON files
        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)
        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}
        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])
            leaf = childs[-1]
            completion_leaf_correct[leaf] = is_correct

        # Function to classify node based on reachable leaves
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_node(node):
            leaves = all_reachable_leaves(node)
            if not leaves:
                return None
            statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child-of-fork nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {child: classify_node(child) for child in child_nodes}

        # Loop over forks
        fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])
        for fork_node in fork_nodes:
            children = list(G.successors(fork_node))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children if c in node_colors]
            # Only consider forks with one red and at least one green/gray
            if "red" not in child_classes:
                continue
            if not any(c in {"green", "gray"} for c in child_classes):
                continue
            if len(set(child_classes)) == 1:
                continue

            for i, child in enumerate(children):
                cls = child_classes[i]
                selected_prob = nodes.get(child, {}).get("selected_prob", None)
                if selected_prob is None:
                    continue
                # Combine green+gray
                if cls in {"green", "gray"}:
                    greengray_probs.append(selected_prob)
                elif cls == "red":
                    red_probs.append(selected_prob)

            # Save paired probabilities for t-test
            red_child_probs = [nodes[c].get("selected_prob") for i, c in enumerate(children)
                               if child_classes[i] == "red" and nodes[c].get("selected_prob") is not None]
            greengray_child_probs = [nodes[c].get("selected_prob") for i, c in enumerate(children)
                                     if child_classes[i] in {"green", "gray"} and nodes[c].get("selected_prob") is not None]

            # Use only one value per fork (e.g., mean if multiple children in same class)
            if red_child_probs and greengray_child_probs:
                paired_probs_red.append(red_child_probs[0])  # or np.mean(red_child_probs)
                paired_probs_greengray.append(greengray_child_probs[0])  # or np.mean(greengray_child_probs)

# Histogram
plt.figure(figsize=(10, 6))
plt.hist(greengray_probs, bins=20, alpha=0.6, label="Correct/Uncertain", color="Green", density=True)
plt.hist(red_probs, bins=20, alpha=0.6, label="Incorrect", color="Red", density=True)
plt.xlabel("Selected Token Probability")
plt.ylabel("Density")
plt.title("Token probabilities for Impactful and Nonimpactful forks")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(greengray_probs, bins=20, alpha=0.6, label="Green/Gray", color="Blue", density=True)
plt.xlabel("Selected Token Probability")
plt.ylabel("Density")
plt.title("Token probabilities for Correct/Uncertain Children of Impactful Forks")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(red_probs, bins=20, alpha=0.6, label="Red", color="Gray", density=True)
plt.xlabel("Selected Token Probability")
plt.ylabel("Density")
plt.title("Token probabilities for Incorrect Children of Impactful Forks")
plt.show()

cats_probs_pivotal = [
    ('greengray_probs', greengray_probs),
    ('red_probs', red_probs)
]

for i in range(len(cats_probs_pivotal)):
    for j in range(i+1, len(cats_probs_pivotal)):
        name1, cat1 = cats_probs_pivotal[i]
        name2, cat2 = cats_probs_pivotal[j]

        wilcoxonstat, wilcoxonp_value = wilcoxon(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Wilcoxon: {wilcoxonstat}, p={wilcoxonp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25  # adjust based on your data

# Cumulative lists
greengray_probs_index = []
red_probs_index = []

# For paired t-test
paired_probs_greengray_index = []
paired_probs_red_index = []

for rollout in range(num_rollouts):
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    # Load JSON files
    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)
    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}
    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])
        leaf = childs[-1]
        completion_leaf_correct[leaf] = is_correct

    # Function to classify node based on reachable leaves
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_node(node):
        leaves = all_reachable_leaves(node)
        if not leaves:
            return None
        statuses = [completion_leaf_correct.get(leaf, True) for leaf in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child-of-fork nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {child: classify_node(child) for child in child_nodes}

    # Loop over forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])
    for fork_node in fork_nodes:
        children = list(G.successors(fork_node))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children if c in node_colors]
        # Only consider forks with one red and at least one green/gray
        if "red" not in child_classes:
            continue
        if not any(c in {"green", "gray"} for c in child_classes):
            continue
        if len(set(child_classes)) == 1:
            continue

        for i, child in enumerate(children):
            cls = child_classes[i]
            node_info = nodes.get(child, {})
            selected_prob = node_info.get("selected_prob", None)
            topk_probs = node_info.get("topk_probs", None)
            if selected_prob is None or topk_probs is None:
                continue

            # Get index of selected_prob in topk_probs
            try:
                selected_index = topk_probs.index(selected_prob)
            except ValueError:
                selected_index = min(range(len(topk_probs)), key=lambda j: abs(topk_probs[j]-selected_prob))

            # Combine green+gray
            if cls in {"green", "gray"}:
                greengray_probs_index.append(selected_index)
            elif cls == "red":
                red_probs_index.append(selected_index)

        # Save paired indices for t-test
        red_indices = [min(range(len(nodes[c].get("topk_probs", []))),
                           key=lambda j: abs(nodes[c]["topk_probs"][j] - nodes[c]["selected_prob"]))
                       for i, c in enumerate(children)
                       if child_classes[i] == "red" and nodes[c].get("selected_prob") is not None and nodes[c].get("topk_probs")]
        greengray_indices = [min(range(len(nodes[c].get("topk_probs", []))),
                                key=lambda j: abs(nodes[c]["topk_probs"][j] - nodes[c]["selected_prob"]))
                             for i, c in enumerate(children)
                             if child_classes[i] in {"green", "gray"} and nodes[c].get("selected_prob") is not None and nodes[c].get("topk_probs")]

        if red_indices and greengray_indices:
            paired_probs_red_index.append(red_indices[0])       # or np.mean(red_indices)
            paired_probs_greengray_index.append(greengray_indices[0])  # or np.mean(greengray_indices)

# Histogram
plt.figure(figsize=(10, 6))
plt.hist(greengray_probs_index, bins=20, alpha=0.6, label="Correct/Uncertain", color="green", density=True)
plt.hist(red_probs_index, bins=20, alpha=0.6, label="Incorrect", color="red", density=True)
plt.xlabel("Selected Token Probability Index")
plt.ylabel("Density")
plt.title("Token probability indices for impactful forks")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(greengray_probs_index, bins=20, alpha=0.6, label="Correct/Uncertain", color="green", density=True)
plt.xlabel("Selected Token Probability Index")
plt.ylabel("Density")
plt.title("Token probability indices for correct/uncertain children of impactful forks")
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(red_probs_index, bins=20, alpha=0.6, label="Red", color="red", density=True)
plt.xlabel("Selected Token Probability Index")
plt.ylabel("Density")
plt.title("Token probability indices for incorrect children of impactful forks")
plt.show()

cats_probs_index_pivotal = [
    ('greengray_probs_index', greengray_probs_index),
    ('red_probs_index', red_probs_index)
]

for i in range(len(cats_probs_index_pivotal)):
    for j in range(i+1, len(cats_probs_index_pivotal)):
        name1, cat1 = cats_probs_index_pivotal[i]
        name2, cat2 = cats_probs_index_pivotal[j]

        wilcoxonstat, wilcoxonp_value = wilcoxon(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Wilcoxon: {wilcoxonstat}, p={wilcoxonp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
#Entropy of Impactful vs Non-Impactful Forks

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25

impactful_entropies = []
nonimpactful_entropies = []

for rollout in range(num_rollouts):

    # Load files
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)

    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])

        completion_leaf_correct[childs[-1]] = is_correct

    # Reachable leaf classification
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_child(node):
        leaves = all_reachable_leaves(node)
        statuses = [completion_leaf_correct.get(l, True) for l in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # All child-of-fork nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {c: classify_child(c) for c in child_nodes}

    # Evaluate forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

    for fork in fork_nodes:
        children = list(G.successors(fork))

        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children]

        has_red = "red" in child_classes
        has_nonred = any(c in {"green", "gray"} for c in child_classes)

        impactful = has_red and has_nonred

        # Collect entropies for all children of this fork
        for child in children:
            entropy = nodes.get(child, {}).get("metric_value")
            if entropy is None:
                continue

            if impactful:
                impactful_entropies.append(entropy)
            else:
                nonimpactful_entropies.append(entropy)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_entropies, bins=25, alpha=0.6, label="Impactful forks", color="blue", density=True)
plt.hist(nonimpactful_entropies, bins=25, alpha=0.6, label="Non-impactful forks", color="gray", density=True)
plt.xlabel("Entropy (metric_value)")
plt.ylabel("Density")
plt.title("Entropy distribution: impactful vs non-impactful forks")
plt.legend()
plt.show()

cats_pivotal_entropy = [
    ('Pivotal Entropies', impactful_entropies),
    ('Non Pivotal Entropies', nonimpactful_entropies)
]

for i in range(len(cats_pivotal_entropy)):
    for j in range(i+1, len(cats_pivotal_entropy)):
        name1, cat1 = cats_pivotal_entropy[i]
        name2, cat2 = cats_pivotal_entropy[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
#Entropy of Impactful vs Non-Impactful Forks

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25

impactful_entropies = []
nonimpactful_entropies = []

for base_dir in base_dirs:
    for rollout in range(num_rollouts):

        # Load files
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)

        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}

        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])

            completion_leaf_correct[childs[-1]] = is_correct

        # Reachable leaf classification
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_child(node):
            leaves = all_reachable_leaves(node)
            statuses = [completion_leaf_correct.get(l, True) for l in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # All child-of-fork nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {c: classify_child(c) for c in child_nodes}

        # Evaluate forks
        fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

        for fork in fork_nodes:
            children = list(G.successors(fork))

            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children]

            has_red = "red" in child_classes
            has_nonred = any(c in {"green", "gray"} for c in child_classes)

            impactful = has_red and has_nonred

            # Collect entropies for all children of this fork
            for child in children:
                entropy = nodes.get(child, {}).get("metric_value")
                if entropy is None:
                    continue

                if impactful:
                    impactful_entropies.append(entropy)
                else:
                    nonimpactful_entropies.append(entropy)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_entropies, bins=25, alpha=0.6, label="Impactful Forks", color="blue", density=True)
plt.hist(nonimpactful_entropies, bins=25, alpha=0.6, label="Non-Impactful Forks", color="gray", density=True)
plt.xlabel("Token Entropy")
plt.ylabel("Density")
plt.title("Entropy: Impactful vs Non-Impactful Forks")
plt.legend()
plt.show()

cats_pivotal_entropy = [
    ('Pivotal Entropies', impactful_entropies),
    ('Non Pivotal Entropies', nonimpactful_entropies)
]

for i in range(len(cats_pivotal_entropy)):
    for j in range(i+1, len(cats_pivotal_entropy)):
        name1, cat1 = cats_pivotal_entropy[i]
        name2, cat2 = cats_pivotal_entropy[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
# Impactful vs Non-Impactful Forks with mean of top k

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25

impactful_topk_means = []
nonimpactful_topk_means = []

for rollout in range(num_rollouts):

    # Load files
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)

    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])

        completion_leaf_correct[childs[-1]] = is_correct

    # Reachable leaf classification
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_child(node):
        leaves = all_reachable_leaves(node)
        statuses = [completion_leaf_correct.get(l, True) for l in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {c: classify_child(c) for c in child_nodes}

    # Evaluate forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

    for fork in fork_nodes:
        children = list(G.successors(fork))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children]

        has_red = "red" in child_classes
        has_nonred = any(c in {"green", "gray"} for c in child_classes)

        impactful = has_red and has_nonred

        # Collect mean(topk_probs) from each child
        for child in children:
            topk_probs = nodes.get(child, {}).get("topk_probs")
            if not topk_probs:
                continue

            mean_topk = float(np.mean(topk_probs))

            if impactful:
                impactful_topk_means.append(mean_topk)
            else:
                nonimpactful_topk_means.append(mean_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_means, bins=25, alpha=0.6, label="Impactful forks", color="blue", density=True)
plt.hist(nonimpactful_topk_means, bins=25, alpha=0.6, label="Non-impactful forks", color="gray", density=True)
plt.xlabel("Mean(topk_probs)")
plt.ylabel("Density")
plt.title("Mean top-k probability: impactful vs non-impactful forks")
plt.legend()
plt.show()

cats_pivotal_mean = [
    ('Pivotal Top k Means', impactful_topk_means),
    ('Non-Pivotal Top k Means', nonimpactful_topk_means)
]

for i in range(len(cats_pivotal_mean)):
    for j in range(i+1, len(cats_pivotal_mean)):
        name1, cat1 = cats_pivotal_mean[i]
        name2, cat2 = cats_pivotal_mean[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
# Impactful vs Non-Impactful Forks with mean of top k all rollouts

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25

impactful_topk_means = []
nonimpactful_topk_means = []


for base_dir in base_dirs:
    for rollout in range(num_rollouts):

        # Load files
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)

        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}

        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])

            completion_leaf_correct[childs[-1]] = is_correct

        # Reachable leaf classification
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_child(node):
            leaves = all_reachable_leaves(node)
            statuses = [completion_leaf_correct.get(l, True) for l in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {c: classify_child(c) for c in child_nodes}

        # Evaluate forks
        fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

        for fork in fork_nodes:
            children = list(G.successors(fork))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children]

            has_red = "red" in child_classes
            has_nonred = any(c in {"green", "gray"} for c in child_classes)

            impactful = has_red and has_nonred

            # Collect mean(topk_probs) from each child
            for child in children:
                topk_probs = nodes.get(child, {}).get("topk_probs")
                if not topk_probs:
                    continue

                mean_topk = float(np.mean(topk_probs))

                if impactful:
                    impactful_topk_means.append(mean_topk)
                else:
                    nonimpactful_topk_means.append(mean_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_means, bins=25, alpha=0.6, label="Impactful Forks", color="blue", density=True)
plt.hist(nonimpactful_topk_means, bins=25, alpha=0.6, label="Non-impactful Forks", color="gray", density=True)
plt.xlabel("Mean(Top 25 Token Probabilities)")
plt.ylabel("Density")
plt.title("Mean of Top 25 Token Probabilities: Impactful vs Non-Impactful Forks")
plt.legend()
plt.show()

cats_pivotal_mean = [
    ('Pivotal Top k Means', impactful_topk_means),
    ('Non-Pivotal Top k Means', nonimpactful_topk_means)
]

for i in range(len(cats_pivotal_mean)):
    for j in range(i+1, len(cats_pivotal_mean)):
        name1, cat1 = cats_pivotal_mean[i]
        name2, cat2 = cats_pivotal_mean[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
# Impactful vs Non-Impactful Forks (using std(topk_probs))

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25

impactful_topk_stds = []
nonimpactful_topk_stds = []

for rollout in range(num_rollouts):

    # Load files
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)

    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])

        completion_leaf_correct[childs[-1]] = is_correct

    # Reachable leaf classification
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_child(node):
        leaves = all_reachable_leaves(node)
        statuses = [completion_leaf_correct.get(l, True) for l in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {c: classify_child(c) for c in child_nodes}

    # Evaluate forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

    for fork in fork_nodes:
        children = list(G.successors(fork))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children]

        has_red = "red" in child_classes
        has_nonred = any(c in {"green", "gray"} for c in child_classes)

        impactful = has_red and has_nonred

        # Collect mean(topk_probs) from each child
        for child in children:
            topk_probs = nodes.get(child, {}).get("topk_probs")
            if not topk_probs:
                continue

            std_topk = float(np.std(topk_probs))

            if impactful:
                impactful_topk_stds.append(std_topk)
            else:
                nonimpactful_topk_stds.append(std_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_stds, bins=25, alpha=0.6, label="Impactful forks", color="blue", density=True)
plt.hist(nonimpactful_topk_stds, bins=25, alpha=0.6, label="Non-impactful forks", color="gray", density=True)
plt.xlabel("Std(topk_probs)")
plt.ylabel("Density")
plt.title("Std top-k probability: impactful vs non-impactful forks")
plt.legend()
plt.show()

cats_pivotal_std = [
    ('Pivotal Top k Stds', impactful_topk_stds),
    ('Non Pivotal Top k Stds', nonimpactful_topk_stds)
]

for i in range(len(cats_pivotal_std)):
    for j in range(i+1, len(cats_pivotal_std)):
        name1, cat1 = cats_pivotal_std[i]
        name2, cat2 = cats_pivotal_std[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
# Impactful vs Non-Impactful Forks (using std(topk_probs)) all rollouts

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25

impactful_topk_stds = []
nonimpactful_topk_stds = []

for base_dir in base_dirs:
    for rollout in range(num_rollouts):

        # Load files
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)

        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}

        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])

            completion_leaf_correct[childs[-1]] = is_correct

        # Reachable leaf classification
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_child(node):
            leaves = all_reachable_leaves(node)
            statuses = [completion_leaf_correct.get(l, True) for l in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {c: classify_child(c) for c in child_nodes}

        # Evaluate forks
        fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

        for fork in fork_nodes:
            children = list(G.successors(fork))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children]

            has_red = "red" in child_classes
            has_nonred = any(c in {"green", "gray"} for c in child_classes)

            impactful = has_red and has_nonred

            # Collect mean(topk_probs) from each child
            for child in children:
                topk_probs = nodes.get(child, {}).get("topk_probs")
                if not topk_probs:
                    continue

                std_topk = float(np.std(topk_probs))

                if impactful:
                    impactful_topk_stds.append(std_topk)
                else:
                    nonimpactful_topk_stds.append(std_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_stds, bins=25, alpha=0.6, label="Impactful Forks", color="blue", density=True)
plt.hist(nonimpactful_topk_stds, bins=25, alpha=0.6, label="Non-impactful Forks", color="gray", density=True)
plt.xlabel("Std(Top 25 Probabilities)")
plt.ylabel("Density")
plt.title("Standard Deviation of Top 25 Probabilities: Impactful vs Non-Impactful Forks")
plt.legend()
plt.show()

cats_pivotal_std = [
    ('Pivotal Top k Stds', impactful_topk_stds),
    ('Non Pivotal Top k Stds', nonimpactful_topk_stds)
]

for i in range(len(cats_pivotal_std)):
    for j in range(i+1, len(cats_pivotal_std)):
        name1, cat1 = cats_pivotal_std[i]
        name2, cat2 = cats_pivotal_std[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
# Impactful vs Non-Impactful Forks (using skew(topk_probs))

from scipy.stats import skew

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25

impactful_topk_skews = []
nonimpactful_topk_skews = []

for rollout in range(num_rollouts):

    # Load files
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)

    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])

        completion_leaf_correct[childs[-1]] = is_correct

    # Reachable leaf classification
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_child(node):
        leaves = all_reachable_leaves(node)
        statuses = [completion_leaf_correct.get(l, True) for l in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {c: classify_child(c) for c in child_nodes}

    # Evaluate forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

    for fork in fork_nodes:
        children = list(G.successors(fork))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children]

        has_red = "red" in child_classes
        has_nonred = any(c in {"green", "gray"} for c in child_classes)

        impactful = has_red and has_nonred

        # Collect mean(topk_probs) from each child
        for child in children:
            topk_probs = nodes.get(child, {}).get("topk_probs")
            if not topk_probs:
                continue

            skew_topk = float(skew(topk_probs))

            if impactful:
                impactful_topk_skews.append(skew_topk)
            else:
                nonimpactful_topk_skews.append(skew_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_skews, bins=25, alpha=0.6, label="Impactful forks", color="blue", density=True)
plt.hist(nonimpactful_topk_skews, bins=25, alpha=0.6, label="Non-impactful forks", color="gray", density=True)
plt.xlabel("Skew(topk_probs)")
plt.ylabel("Density")
plt.title("Skew top-k probability: impactful vs non-impactful forks")
plt.legend()
plt.show()

cats_pivotal_skew = [
    ('Pivotal Top k Skews', impactful_topk_skews),
    ('Non Pivotal Top k Skews', nonimpactful_topk_skews)
]

for i in range(len(cats_pivotal_skew)):
    for j in range(i+1, len(cats_pivotal_skew)):
        name1, cat1 = cats_pivotal_skew[i]
        name2, cat2 = cats_pivotal_skew[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
# Impactful vs Non-Impactful Forks (using skew(topk_probs))

from scipy.stats import skew

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25

impactful_topk_skews = []
nonimpactful_topk_skews = []

for base_dir in base_dirs:
    for rollout in range(num_rollouts):

        # Load files
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)

        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}

        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])

            completion_leaf_correct[childs[-1]] = is_correct

        # Reachable leaf classification
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_child(node):
            leaves = all_reachable_leaves(node)
            statuses = [completion_leaf_correct.get(l, True) for l in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {c: classify_child(c) for c in child_nodes}

        # Evaluate forks
        fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

        for fork in fork_nodes:
            children = list(G.successors(fork))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children]

            has_red = "red" in child_classes
            has_nonred = any(c in {"green", "gray"} for c in child_classes)

            impactful = has_red and has_nonred

            # Collect mean(topk_probs) from each child
            for child in children:
                topk_probs = nodes.get(child, {}).get("topk_probs")
                if not topk_probs:
                    continue

                skew_topk = float(skew(topk_probs))

                if impactful:
                    impactful_topk_skews.append(skew_topk)
                else:
                    nonimpactful_topk_skews.append(skew_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_skews, bins=25, alpha=0.6, label="Impactful forks", color="blue", density=True)
plt.hist(nonimpactful_topk_skews, bins=25, alpha=0.6, label="Non-impactful forks", color="gray", density=True)
plt.xlabel("Skew(topk_probs)")
plt.ylabel("Density")
plt.title("Skew top-k probability: impactful vs non-impactful forks")
plt.legend()
plt.show()

cats_pivotal_skew = [
    ('Pivotal Top k Skews', impactful_topk_skews),
    ('Non Pivotal Top k Skews', nonimpactful_topk_skews)
]

for i in range(len(cats_pivotal_skew)):
    for j in range(i+1, len(cats_pivotal_skew)):
        name1, cat1 = cats_pivotal_skew[i]
        name2, cat2 = cats_pivotal_skew[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )

In [ ]:
# Impactful vs Non-Impactful Forks (using max(topk_probs))

base_dir = "/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12"
num_rollouts = 25

impactful_topk_max = []
nonimpactful_topk_max = []

for rollout in range(num_rollouts):

    # Load files
    paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
    graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

    with open(paths_file, "r") as f:
        completions = json.load(f)
    with open(graph_file, "r") as f:
        graph = json.load(f)

    nodes = graph["graph"]["nodes"]

    # Build DAG
    G = nx.DiGraph()
    completion_leaf_correct = {}

    for comp in completions:
        forks = comp["forking_node_ids"]
        childs = comp["child_of_fork"]
        is_correct = comp["correct"]

        for i in range(len(forks)):
            G.add_edge(forks[i], childs[i])
            if i + 1 < len(forks):
                G.add_edge(childs[i], forks[i + 1])

        completion_leaf_correct[childs[-1]] = is_correct

    # Reachable leaf classification
    def all_reachable_leaves(node):
        leaves = set()
        stack = [node]
        while stack:
            n = stack.pop()
            children = list(G.successors(n))
            if not children:
                leaves.add(n)
            else:
                stack.extend(children)
        return leaves

    def classify_child(node):
        leaves = all_reachable_leaves(node)
        statuses = [completion_leaf_correct.get(l, True) for l in leaves]
        if all(statuses):
            return "green"
        elif all(not s for s in statuses):
            return "red"
        else:
            return "gray"

    # Collect child nodes
    child_nodes = set()
    for comp in completions:
        child_nodes.update(comp["child_of_fork"])

    node_colors = {c: classify_child(c) for c in child_nodes}

    # Evaluate forks
    fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

    for fork in fork_nodes:
        children = list(G.successors(fork))
        if not children:
            continue

        child_classes = [node_colors.get(c) for c in children]

        has_red = "red" in child_classes
        has_nonred = any(c in {"green", "gray"} for c in child_classes)

        impactful = has_red and has_nonred

        # Collect mean(topk_probs) from each child
        for child in children:
            topk_probs = nodes.get(child, {}).get("topk_probs")
            if not topk_probs:
                continue

            max_topk = float(np.max(topk_probs))

            if impactful:
                impactful_topk_max.append(max_topk)
            else:
                nonimpactful_topk_max.append(max_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_max, bins=25, alpha=0.6, label="Impactful forks", color="blue", density=True)
plt.hist(nonimpactful_topk_max, bins=25, alpha=0.6, label="Non-impactful forks", color="gray", density=True)
plt.xlabel("Max(topk_probs)")
plt.ylabel("Density")
plt.title("Max top-k probability: impactful vs non-impactful forks")
plt.legend()
plt.show()

cats_pivotal_max = [
    ('Pivotal Max Prob', impactful_topk_max),
    ('NonPivotal Max Prob', nonimpactful_topk_max)
]

for i in range(len(cats_pivotal_max)):
    for j in range(i+1, len(cats_pivotal_max)):
        name1, cat1 = cats_pivotal_max[i]
        name2, cat2 = cats_pivotal_max[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
# Impactful vs Non-Impactful Forks (using max(topk_probs)) for all rollouts

base_dirs = ["/home/tomg1018/MAT1510-Project/tom1510_20251123_191006/tom1510_20251123_191006/problem_12","/home/tomg1018/MAT1510-Project/tom1510_20251123_214229/tom1510_20251123_214229/problem_9", "/home/tomg1018/MAT1510-Project/tom1510_20251123_214417/tom1510_20251123_214417/problem_0"]
num_rollouts = 25

impactful_topk_max = []
nonimpactful_topk_max = []

for base_dir in base_dirs:
    for rollout in range(num_rollouts):

        # Load files
        paths_file = os.path.join(base_dir, f"rollout_{rollout}", "paths.json")
        graph_file = os.path.join(base_dir, f"rollout_{rollout}", "graph_structure.json")

        with open(paths_file, "r") as f:
            completions = json.load(f)
        with open(graph_file, "r") as f:
            graph = json.load(f)

        nodes = graph["graph"]["nodes"]

        # Build DAG
        G = nx.DiGraph()
        completion_leaf_correct = {}

        for comp in completions:
            forks = comp["forking_node_ids"]
            childs = comp["child_of_fork"]
            is_correct = comp["correct"]

            for i in range(len(forks)):
                G.add_edge(forks[i], childs[i])
                if i + 1 < len(forks):
                    G.add_edge(childs[i], forks[i + 1])

            completion_leaf_correct[childs[-1]] = is_correct

        # Reachable leaf classification
        def all_reachable_leaves(node):
            leaves = set()
            stack = [node]
            while stack:
                n = stack.pop()
                children = list(G.successors(n))
                if not children:
                    leaves.add(n)
                else:
                    stack.extend(children)
            return leaves

        def classify_child(node):
            leaves = all_reachable_leaves(node)
            statuses = [completion_leaf_correct.get(l, True) for l in leaves]
            if all(statuses):
                return "green"
            elif all(not s for s in statuses):
                return "red"
            else:
                return "gray"

        # Collect child nodes
        child_nodes = set()
        for comp in completions:
            child_nodes.update(comp["child_of_fork"])

        node_colors = {c: classify_child(c) for c in child_nodes}

        # Evaluate forks
        fork_nodes = set(f for comp in completions for f in comp["forking_node_ids"])

        for fork in fork_nodes:
            children = list(G.successors(fork))
            if not children:
                continue

            child_classes = [node_colors.get(c) for c in children]

            has_red = "red" in child_classes
            has_nonred = any(c in {"green", "gray"} for c in child_classes)

            impactful = has_red and has_nonred

            # Collect mean(topk_probs) from each child
            for child in children:
                topk_probs = nodes.get(child, {}).get("topk_probs")
                if not topk_probs:
                    continue

                max_topk = float(np.max(topk_probs))

                if impactful:
                    impactful_topk_max.append(max_topk)
                else:
                    nonimpactful_topk_max.append(max_topk)


# ----------- Plotting ----------------

plt.figure(figsize=(10,6))
plt.hist(impactful_topk_max, bins=25, alpha=0.6, label="Impactful Forks", color="blue", density=True)
plt.hist(nonimpactful_topk_max, bins=25, alpha=0.6, label="Non-Impactful Forks", color="gray", density=True)
plt.xlabel("Max(Top 25 Token Probabilities)")
plt.ylabel("Density")
plt.title("Max of Top 25 Token Probabilities: Impactful vs Non-Impactful Forks")
plt.legend()
plt.show()

cats_pivotal_max = [
    ('Pivotal Max Prob', impactful_topk_max),
    ('NonPivotal Max Prob', nonimpactful_topk_max)
]

for i in range(len(cats_pivotal_max)):
    for j in range(i+1, len(cats_pivotal_max)):
        name1, cat1 = cats_pivotal_max[i]
        name2, cat2 = cats_pivotal_max[j]

        mannwhitneystat, mannwhitneyp_value = mannwhitneyu(cat1, cat2, alternative="two-sided")
        ksstat, ksp_value = ks_2samp(cat1, cat2)

        print(
            f"{name1} vs {name2} — "
            f"Mann Whitney U: {mannwhitneystat}, p={mannwhitneyp_value}; "
            f"KS: {ksstat}, p={ksp_value}"
        )


In [ ]:
import numpy as np

entropy_green, entropy_red, entropy_gray = np.array(entropy_green_children), np.array(entropy_red_children), np.array(entropy_gray_children)
entropy_correct_so_far = np.concatenate((entropy_green, entropy_gray))

prob_green, prob_red, prob_gray = np.array(prob_green_children), np.array(prob_red_children), np.array(prob_gray_children)
prob_correct_so_far = np.concatenate((prob_green, prob_gray))

prob_index_green, prob_index_red, prob_index_gray = np.array(prob_index_green_children), np.array(prob_index_red_children), np.array(prob_index_gray_children)
prob_index_correct_so_far = np.concatenate((prob_index_green, prob_index_gray))

stat, p_value = mannwhitneyu(entropy_green, entropy_red, alternative = "two-sided")
print(stat, p_value)